In [46]:
# Load the Kedro IPython extension
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [47]:
# Read parquet file into a DataFrame with polars
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [58]:
df = catalog.load("second_stage_predictions_tct")
df.write_csv(
    "/Users/Carlos_Davalos/Library/CloudStorage/OneDrive-McKinsey&Company/Documents/COPEC/resultados/second_stage_predictions_tct.csv"
)

[07/03/25 22:01:24] INFO     Loading data from second_stage_predictions_tct                     ]8;id=782772;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=87365;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\
                             (PolarsParquetDataset)...                                                             

In [48]:
# df = pl.read_parquet(
#     "/Users/Carlos_Davalos/Downloads/w_0.8_d_60_o_3.0_t_0.01/second_stage_predictions.parquet"
# ).to_pandas()

df = catalog.load("second_stage_predictions_tct").to_pandas()

df["corrected_predicted_value"] = df.apply(
    lambda row: row["predicted_value"]
    if row["predicted_value"] > row["c_yearly_margin_per_liter"]
    else row["c_yearly_margin_per_liter"],
    axis=1,
)

df["margin_change"] = df["corrected_predicted_value"] - df["c_yearly_margin_per_liter"]


def forced_decile_binning(series: pd.Series, prefix="Q"):
    # Perform quantile-based binning with duplicate edges dropped
    binned, bin_edges = pd.qcut(series, q=10, retbins=True, duplicates="drop")

    # Determine actual number of bins (may be < 10 if duplicates dropped)
    num_bins = len(bin_edges) - 1
    labels = [f"{prefix}{i + 1}" for i in range(num_bins)]

    # Re-bin using the known edges with labels applied
    binned = pd.cut(series, bins=bin_edges, labels=labels, include_lowest=True)

    # Create mapping from label to interval
    mapping = {labels[i]: (bin_edges[i], bin_edges[i + 1]) for i in range(num_bins)}

    return binned, mapping


# Apply to your DataFrame
df["margin_quantile"], quantile_label_map = forced_decile_binning(
    df["corrected_predicted_value"]
)


def assign_quantiles(df, column, quantiles=10):
    # Define labels dynamically based on column name
    labels = [f"Q{i + 1}" for i in range(quantiles - 1)]

    # Create the new column name
    new_col_name = f"{column}_quantile"

    # Assign quantiles
    df[new_col_name] = pd.qcut(df[column], q=quantiles, duplicates="drop")

    return df


df = assign_quantiles(df, "c_yearly_network_volumen")
df = assign_quantiles(df, "c_weighted_competitive_index")

columns_to_quantile = [
    "share_customer_region_1",
    "share_customer_region_2",
    "share_customer_region_3",
    "share_customer_region_4",
    "share_customer_region_5",
    "share_customer_region_6",
    "share_customer_region_7",
    "share_customer_region_8",
    "share_customer_region_9",
    "share_customer_region_10",
    "share_customer_region_11",
    "share_customer_region_12",
    "share_customer_region_13",
    "share_customer_region_14",
    "share_customer_region_15",
    "share_customer_region_16",
]


# def apply_forced_binning(df, columns):
#     binning_mappings = {}

#     for col in columns:
#         prefix = "Q"
#         binned, mapping = forced_decile_binning(df[col], prefix=prefix)
#         df[f"{col}_quantile"] = binned
#         binning_mappings[col] = mapping

#     return df, binning_mappings


# # df, quantile_mappings = apply_forced_binning(df, columns_to_quantile)

[07/03/25 21:45:58] INFO     Loading data from second_stage_predictions_tct                     ]8;id=339939;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=139740;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\
                             (PolarsParquetDataset)...                                                             

In [49]:
for col in columns_to_quantile:
    df = assign_quantiles(df, col)

In [50]:
def format_interval_as_percent(interval):
    # Convert pandas.Interval to percentage string, e.g., (0.07, 0.11] → "7%–11%"
    left = f"{interval.left * 100:.0f}%"
    right = f"{interval.right * 100:.0f}%"
    return f"{left}–{right}"


def format_millions(value):
    return f"{value / 1000000:.1f}M" if value >= 1000000 else f"{value:,.0f}"


def format_volume_interval(interval):
    left = format_millions(interval.left)
    right = format_millions(interval.right)
    return f"{left}–{right}"

In [51]:
# Stats summary
total = len(df)
top_percent = (df["performance_label"] == "Top Performer").sum() / total * 100
under_percent = (df["performance_label"] == "Under Performer").sum() / total * 100
summary_text = f"✅ Top Performers: {top_percent:.1f}%<br>❌ Under Performers: {under_percent:.1f}%"

# Create vertical subplots (scatter on top, histogram below)
fig = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.15,
    specs=[[{"type": "scatter"}], [{"type": "xy"}]],
)

# Scatter plot: predicted value
scatter_pred = px.scatter(
    df,
    x="c_yearly_margin_per_liter",
    y="corrected_predicted_value",
    size="c_yearly_network_volumen",
    color="performance_label",
    size_max=20,
)
scatter_pred.update_traces(marker=dict(line=dict(width=0)))
for trace in scatter_pred.data:
    fig.add_trace(trace, row=1, col=1)

# Scatter plot: corrected predicted value
scatter_corr = px.scatter(
    df,
    x="c_yearly_margin_per_liter",
    y="corrected_predicted_value",
    size="c_yearly_network_volumen",
    color="performance_label",
    size_max=20,
)
scatter_corr.update_traces(
    marker=dict(line=dict(width=0), opacity=0.6, symbol="circle-open")
)

# Histogram: margin change
fig.add_trace(
    go.Histogram(
        x=df["margin_change"],
        nbinsx=30,
        marker_color="teal",
        opacity=0.75,
        showlegend=False,
        histnorm="probability",
    ),
    row=2,
    col=1,
)

# Annotations
fig.add_annotation(
    text="Predicted vs Actual Margin",
    x=0.5,
    y=1.08,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=16),
    xanchor="center",
)
fig.add_annotation(
    text="Relative Error Distribution",
    x=0.5,
    y=0.35,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=14),
    xanchor="center",
)

# Performance summary box
fig.add_annotation(
    text=summary_text,
    xref="paper",
    yref="paper",
    x=0.01,
    y=0.96,
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12),
)

# Layout updates
fig.update_layout(
    height=700,
    title_text="Performance Overview",
    margin=dict(t=100, b=60, l=60, r=40),
    showlegend=True,
)

# Axis labels
fig.update_xaxes(title_text="Actual Margin (c_yearly_margin_per_liter)", row=1, col=1)
fig.update_yaxes(title_text="Predicted Margin", row=1, col=1)
fig.update_xaxes(title_text="Relative Margin Change", row=2, col=1)
fig.update_yaxes(title_text="Probability", row=2, col=1)

fig.show()


In [52]:
import dash
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Input, Output, dcc, html
from plotly.subplots import make_subplots

# Dummy data example (replace this with your real `df`)
# df = pd.read_csv("your_data.csv")
# Example structure (you'll replace with actual df)
# df["margin_quantile"] = pd.qcut(df["margin_change"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

# df["margin_quantile"] = pd.qcut(df["margin_change"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

# Assuming df is already available
# numeric_cols = [col for col in df.select_dtypes(include="number").columns]
numeric_cols = df.columns

app = dash.Dash(__name__)
app.title = "Interactive Dashboard"

# --- 1. Predicted vs. Actual Margin ---
total = len(df)
top_percent = (df["performance_label"] == "Top Performer").sum() / total * 100
under_percent = (df["performance_label"] == "Under Performer").sum() / total * 100
non_regular_percent = (
    (df["performance_label"] == "Non Regular Client").sum() / total * 100
)
impact = sum(df["margin_change"] * df["c_yearly_volumen"]) * 0.0011
impact_millions = impact / 1_000_000

summary_text = (
    f"<b>📊 Potencial Total: ${impact_millions:,.2f}M</b><br>"
    f"<b>📊 Numero de Records: {total}</b><br>"
    f"✅ Top Performers: {top_percent:.1f}%<br>"
    f"❌ Under Performers: {under_percent:.1f}%"
    f"<br>⚠️ Non Regular Clients: {non_regular_percent:.1f}%<br>"
)

scatter_pred = px.scatter(
    df,
    x="c_yearly_margin_per_liter",
    y="corrected_predicted_value",
    size="c_yearly_network_volumen",
    color="performance_label",
    size_max=20,
    hover_data={
        "customer_id": True,
        "c_yearly_margin_per_liter": ":$,.0f",  # Format as currency
        "corrected_predicted_value": ":$,.0f",  # Format as currency, no decimals
        "c_yearly_network_volumen": ":,.0f",  # Comma-separated large numbers
    },
)

scatter_pred.update_layout(
    title="Se Predice el Margen Anual por Litro. En caso de que la prediccion sea menor al margen anual por litro, se corrige a este ultimo.",
    xaxis_title="Margen Anual por Litro (Actual)",
    yaxis_title="Margen Anual por Litro (Prediccion)",
    bargap=0.05,
)

scatter_pred.update_traces(marker=dict(line=dict(width=0)))

scatter_pred.add_annotation(
    text=summary_text,
    xref="paper",
    yref="paper",
    x=0.01,
    y=0.99,
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12),
)

# --- 2. Histogram of Margin Change ---
# Compute min and max with rounding for bin alignment
start = df["margin_change"].min() // 1 * 1
end = df["margin_change"].max() // 1 * 1 + 1

df["margin_binned"] = df["margin_change"].copy()

# Clip all values greater than 20 to exactly 20
df["margin_binned"] = df["margin_binned"].apply(lambda x: 20 if x > 20 else x)

fig_hist = go.Figure()

fig_hist.add_trace(
    go.Histogram(
        x=df["margin_binned"],
        xbins=dict(
            start=df["margin_binned"].min() // 1 * 1,
            end=25,  # Include overflow bin up to 25
            size=1,
        ),
        marker_color="teal",
        opacity=0.75,
        showlegend=False,
    )
)

fig_hist.update_layout(
    title="Cantidad de Clientes por Cambio de Margen (Prediccion - Actual), (Agrupamos >20)",
    xaxis_title="Cambio de Margen (Prediccion - Actual)",
    yaxis_title="Frecuencia",
    bargap=0.05,
)

# --- 3. Volume vs Margin Scatterplot ---
fig_volume_margin = px.scatter(
    df,
    x="c_yearly_network_volumen",
    y="margin_change",
    color="performance_label",
    size_max=20,
    title="Volume vs. Margin Change",
    hover_data={
        "customer_id": True,
        "margin_change": ":$,.0f",  # Format as currency
        "c_yearly_network_volumen": ":,.0f",  # Comma-separated large numbers
    },
)

fig_volume_margin.update_layout(
    title="Evaluacion de Clientes por Volumen Anual vs Cambio de Margen",
    xaxis_title="Volumen Anual (Litros)",
    yaxis_title="Cambio de Margen (Prediccion - Actual)",
    bargap=0.05,
)


# App layout
app.layout = html.Div(
    [
        html.H1("Reporte - Modelo B2B Pricing"),
        html.H2("1. Margen - Prediccion vs Actual"),
        dcc.Graph(figure=scatter_pred),
        html.H2("2. Histograma de Cambio de Margen"),
        dcc.Graph(figure=fig_hist),
        html.H2("3. Volumen (Litros) vs Cambio de Margen"),
        dcc.Graph(figure=fig_volume_margin),
        html.H2("4. Histograma por Agrupaciones"),
        html.Div(
            [
                html.Label("Seleccionar columna de Analisis:"),
                dcc.Dropdown(
                    id="single-quantile-col",
                    options=[
                        {"label": col, "value": col} for col in columns_to_quantile
                    ],
                    value=columns_to_quantile[0],
                ),
                html.Label("Seleccionar variable continua:"),
                dcc.Dropdown(
                    id="single-histogram-var",
                    options=[{"label": col, "value": col} for col in numeric_cols],
                    value="c_yearly_network_volumen",
                ),
            ],
            style={"margin-bottom": "20px"},
        ),
        dcc.Graph(id="heatmap_avg_prediction"),
        dcc.Graph(id="heatmap_client_count"),
    ]
)


@app.callback(
    Output("heatmap_avg_prediction", "figure"),
    Output("heatmap_client_count", "figure"),
    Input("single-quantile-col", "value"),
    Input("single-histogram-var", "value"),  # Not used, but can trigger refresh
)
def update_heatmaps(selected_quantile_col, _):
    df_ = df[df["performance_label"] == "Top Performer"]
    df_copy = (
        df_[
            [
                "c_yearly_network_volumen",
                "c_yearly_margin_per_liter",
                "corrected_predicted_value",
                selected_quantile_col,
            ]
        ]
        .dropna()
        .copy()
    )

    # Compute dynamic quantile bins
    try:
        # df_copy["volume_quantile"] = pd.qcut(df_copy["c_yearly_network_volumen"], 10)
        # Step 1: Sort and compute cumulative volume
        df_sorted = df_copy.sort_values("c_yearly_network_volumen").copy()
        df_sorted["cumsum_volume"] = df_sorted["c_yearly_network_volumen"].cumsum()
        total_volume = df_sorted["c_yearly_network_volumen"].sum()
        df_sorted["volume_share"] = df_sorted["cumsum_volume"] / total_volume

        # Step 2: Assign bin number based on cumulative volume
        df_sorted["volume_bin"] = (df_sorted["volume_share"] * 10).apply(
            np.floor
        ).astype(int) + 1
        df_sorted["volume_bin"] = df_sorted["volume_bin"].clip(upper=10)

        # Step 3: Create intervals for labels
        bin_edges_df = df_sorted.groupby("volume_bin")["c_yearly_network_volumen"].agg(
            ["min", "max"]
        )
        bin_intervals = bin_edges_df.apply(
            lambda row: pd.Interval(left=row["min"], right=row["max"], closed="right"),
            axis=1,
        )

        # Step 4: Map bin number to interval and assign as Categorical
        df_sorted["volume_quantile"] = df_sorted["volume_bin"].map(bin_intervals)

        # Convert to ordered categorical so `.cat.*` works
        cat_type = pd.api.types.CategoricalDtype(
            categories=bin_intervals.tolist(), ordered=True
        )
        df_sorted["volume_quantile"] = df_sorted["volume_quantile"].astype(cat_type)

        # Step 5: Merge into df_copy
        df_copy = df_copy.merge(
            df_sorted[["c_yearly_network_volumen", "volume_quantile"]],
            on="c_yearly_network_volumen",
            how="left",
        )
        df_copy["decile"] = pd.qcut(
            df_copy[selected_quantile_col], 10, duplicates="drop"
        )
    except ValueError:
        return go.Figure(), go.Figure()  # Handle edge cases gracefully

    # Convert bin categories to strings and preserve order
    volume_categories = df_copy["volume_quantile"].cat.categories
    decile_categories = df_copy["decile"].cat.categories

    ordered_volume_labels = [format_volume_interval(c) for c in volume_categories]
    ordered_decile_labels = [format_interval_as_percent(c) for c in decile_categories]

    df_copy["volume_quantile"] = pd.Categorical(
        [format_volume_interval(c) for c in df_copy["volume_quantile"]],
        categories=ordered_volume_labels,
        ordered=True,
    )
    df_copy["quantile_group"] = pd.Categorical(
        [format_interval_as_percent(c) for c in df_copy["decile"]],
        categories=ordered_decile_labels,
        ordered=True,
    )

    # --- 1. Avg predicted value heatmap ---
    avg_df = (
        df_copy.groupby(["quantile_group", "volume_quantile"], observed=True)
        .agg(avg_value=("corrected_predicted_value", "mean"))
        .reset_index()
    )
    pivot_avg = avg_df.pivot(
        index="quantile_group", columns="volume_quantile", values="avg_value"
    )

    fig_avg = px.imshow(
        pivot_avg,
        labels=dict(
            x="Cuantil de Volumen",
            y="Agrupacion de Variable",
            color="Predicción Promedio",
        ),
        x=ordered_volume_labels,
        y=ordered_decile_labels,
        text_auto=".2f",
        aspect="auto",
    )
    fig_avg.update_layout(
        title=f"Predicción Promedio por Agrupacion de {selected_quantile_col} y Agrupacion de Volumen"
    )

    # --- 2. Client count heatmap ---
    count_df = (
        df_copy.groupby(["quantile_group", "volume_quantile"], observed=True)
        .size()
        .reset_index(name="client_count")
    )
    pivot_count = count_df.pivot(
        index="quantile_group", columns="volume_quantile", values="client_count"
    )

    fig_count = px.imshow(
        pivot_count,
        labels=dict(
            x="Agrupacion de Volumen",
            y="Agrupacion de Variable",
            color="Cantidad de Clientes",
        ),
        x=ordered_volume_labels,
        y=ordered_decile_labels,
        text_auto=True,
        aspect="auto",
    )
    fig_count.update_layout(
        title=f"Cantidad de Clientes por Agrupacion de {selected_quantile_col} y Agrupacion de Volumen"
    )

    return fig_avg, fig_count


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=9191)

<IPython.lib.display.IFrame object at 0x3239a8c10>

In [53]:
import dash
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Input, Output, dcc, html

numeric_cols = df.select_dtypes(include="number").columns.tolist()
columns_to_quantile = ["c_yearly_margin_per_liter", "margin_change"]  # For future use

app = dash.Dash(__name__)
app.title = "Interactive Dashboard"

# --- App Layout ---
app.layout = html.Div(
    [
        html.H1("Reporte - Modelo B2B Pricing"),
        html.H2("Filtros Globales"),
        html.Label("Seleccionar categorías de riesgo:"),
        dcc.Checklist(
            id="risk-category-filter",
            options=[
                {"label": cat, "value": cat} for cat in df["risk_category"].unique()
            ],
            value=["safe", "top_performer"],
            labelStyle={"display": "inline-block", "margin-right": "10px"},
        ),
        html.H2("1. Margen - Prediccion vs Actual"),
        dcc.Graph(id="scatter-pred"),
        html.H2("2. Histograma de Cambio de Margen"),
        dcc.Graph(id="histogram-margin"),
        html.H2("3. Variable Continua vs Margen Objetivo"),
        html.Label("Seleccionar variable continua para eje X:"),
        dcc.Dropdown(
            id="x-axis-variable",
            options=[
                {"label": col.replace("_", " ").title(), "value": col}
                for col in numeric_cols
                if col != "margin_change"
            ],
            value="c_yearly_network_volumen",
        ),
        dcc.Graph(id="volume-vs-margin"),
    ]
)


# --- Callback ---
@app.callback(
    Output("scatter-pred", "figure"),
    Output("histogram-margin", "figure"),
    Output("volume-vs-margin", "figure"),
    Input("risk-category-filter", "value"),
    Input("x-axis-variable", "value"),
)
def update_static_graphs(selected_risks, x_var):
    df_filtered = df[df["risk_category"].isin(selected_risks)].copy()

    total = len(df_filtered)
    top_percent = (
        ((df_filtered["performance_label"] == "Top Performer").sum() / total * 100)
        if total > 0
        else 0
    )
    under_percent = (
        ((df_filtered["performance_label"] == "Under Performer").sum() / total * 100)
        if total > 0
        else 0
    )
    non_regular_percent = (
        ((df_filtered["performance_label"] == "Non Regular Client").sum() / total * 100)
        if total > 0
        else 0
    )
    impact = (
        sum(df_filtered["margin_change"] * df_filtered["c_yearly_volumen"]) * 0.0011
        if total > 0
        else 0
    )
    impact_millions = impact / 1_000_000

    summary_text = (
        f"<b>📊 Potencial Total: ${impact_millions:,.2f}M</b><br>"
        f"<b>📊 Numero de Records: {total}</b><br>"
        f"✅ Top Performers: {top_percent:.1f}%<br>"
        f"❌ Under Performers: {under_percent:.1f}%"
        f"<br>⚠️ Non Regular Clients: {non_regular_percent:.1f}%<br>"
    )

    # 1. Scatter Pred vs Actual
    scatter_pred = px.scatter(
        df_filtered,
        x="c_yearly_margin_per_liter",
        y="corrected_predicted_value",
        size="c_yearly_network_volumen",
        color="performance_label",
        size_max=20,
        hover_data={
            "customer_id": True,
            "c_yearly_margin_per_liter": ":$,.0f",
            "corrected_predicted_value": ":$,.0f",
            "c_yearly_network_volumen": ":,.0f",
        },
    )
    scatter_pred.update_layout(
        title="Margen - Prediccion vs Actual",
        xaxis_title="Margen Actual",
        yaxis_title="Margen Predicho",
    )
    scatter_pred.update_traces(marker=dict(line=dict(width=0)))
    scatter_pred.add_annotation(
        text=summary_text,
        xref="paper",
        yref="paper",
        x=0.01,
        y=0.99,
        showarrow=False,
        align="left",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=12),
    )

    # 2. Histogram of Margin Change
    df_filtered["margin_binned"] = df_filtered["margin_change"].apply(
        lambda x: 20 if x > 20 else x
    )
    fig_hist = go.Figure()
    fig_hist.add_trace(
        go.Histogram(
            x=df_filtered["margin_binned"],
            xbins=dict(start=0, end=25, size=1),
            marker_color="teal",
            opacity=0.75,
            showlegend=False,
        )
    )
    fig_hist.update_layout(
        title="Histograma de Cambio de Margen",
        xaxis_title="Cambio de Margen",
        yaxis_title="Frecuencia",
    )

    # 3. Custom X-axis vs Margin Change
    fig_volume_margin = px.scatter(
        df_filtered,
        x=x_var,
        y="predicted_value",
        color="performance_label",
        size_max=20,
        hover_data={
            "customer_id": True,
            "predicted_value": ":$,.0f",
            x_var: ":,.0f",
        },
    )
    fig_volume_margin.update_layout(
        title=f"{x_var.replace('_', ' ').title()} vs Margen Objetivo",
        xaxis_title=x_var.replace("_", " ").title(),
        yaxis_title="Margen Objetivo",
    )

    return scatter_pred, fig_hist, fig_volume_margin


# --- Run App ---
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=9191, debug=True)


<IPython.lib.display.IFrame object at 0x3187b2650>

In [54]:
df

,customer_id,c_yearly_volumen,c_yearly_charged_amount,c_yearly_margin,c_yearly_trx_count,c_yearly_regions_count,c_yearly_stations_count,c_days_since_last_trx,c_yearly_avg_volume_per_trx,c_yearly_charged_amount_per_liter,...,share_customer_region_8_quantile,share_customer_region_9_quantile,share_customer_region_10_quantile,share_customer_region_11_quantile,share_customer_region_12_quantile,share_customer_region_13_quantile,share_customer_region_14_quantile,share_customer_region_15_quantile,share_customer_region_16_quantile,margin_binned
0,id_10039613,103771.27,1.060558e+08,7.872003e+06,431,2,4,0,240.768608,1022.014860,...,"(-0.001, 0.00136]","(-0.001, 0.00641]","(-0.001, 0.00685]","(-0.001, 1.0]","(-0.001, 1.0]","(-0.001, 0.0116]","(-0.001, 0.000642]","(-0.001, 1.0]","(-0.001, 0.0064]",2.348814
1,id_10060685,26804.80,2.762214e+07,2.032689e+06,83,4,9,6,322.949398,1030.492337,...,"(-0.001, 0.00136]","(-0.001, 0.00641]","(-0.001, 0.00685]","(-0.001, 1.0]","(-0.001, 1.0]","(0.874, 1.0]","(-0.001, 0.000642]","(-0.001, 1.0]","(-0.001, 0.0064]",1.630697
2,id_10070136,52207.27,5.184360e+07,3.461502e+06,179,3,8,0,291.660726,993.034131,...,"(0.494, 1.0]","(-0.001, 0.00641]","(-0.001, 0.00685]","(-0.001, 1.0]","(-0.001, 1.0]","(-0.001, 0.0116]","(-0.001, 0.000642]","(-0.001, 1.0]","(-0.001, 0.0064]",6.133661
3,id_10194336,73972.16,7.106965e+07,2.714894e+06,139,7,19,5,532.173813,960.762130,...,"(0.0205, 0.083]","(0.00641, 0.0306]","(0.00685, 0.0665]","(-0.001, 1.0]","(-0.001, 1.0]","(0.468, 0.874]","(0.0178, 1.0]","(-0.001, 1.0]","(-0.001, 0.0064]",20.000000
4,id_10323047,35020.58,3.536756e+07,2.326681e+06,153,4,18,3,228.892680,1009.907774,...,"(0.00136, 0.0205]","(-0.001, 0.00641]","(-0.001, 0.00685]","(-0.001, 1.0]","(-0.001, 1.0]","(-0.001, 0.0116]","(-0.001, 0.000642]","(-0.001, 1.0]","(0.0064, 0.0259]",3.360001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8025,idg_006208,316204.90,3.297581e+08,2.142805e+07,268,4,12,0,1179.869030,1042.862087,...,"(0.00136, 0.0205]","(-0.001, 0.00641]","(-0.001, 0.00685]","(-0.001, 1.0]","(-0.001, 1.0]","(-0.001, 0.0116]","(0.000642, 0.0178]","(-0.001, 1.0]","(-0.001, 0.0064]",0.000000
8026,idg_006221,75662.80,7.433529e+07,4.345902e+06,186,9,41,0,406.789247,982.454866,...,"(0.0205, 0.083]","(0.00641, 0.0306]","(-0.001, 0.00685]","(-0.001, 1.0]","(-0.001, 1.0]","(0.468, 0.874]","(-0.001, 0.000642]","(-0.001, 1.0]","(0.0064, 0.0259]",0.000000
8027,idg_006223,1799391.64,1.723634e+09,6.561003e+07,4687,6,67,0,383.911167,957.898455,...,"(0.494, 1.0]","(-0.001, 0.00641]","(-0.001, 0.00685]","(-0.001, 1.0]","(-0.001, 1.0]","(-0.001, 0.0116]","(-0.001, 0.000642]","(-0.001, 1.0]","(0.0823, 1.0]",0.000000
8028,idg_006252,210726.56,2.104293e+08,1.109661e+07,1144,5,47,0,184.201538,998.589347,...,"(-0.001, 0.00136]","(-0.001, 0.00641]","(-0.001, 0.00685]","(-0.001, 1.0]","(-0.001, 1.0]","(0.468, 0.874]","(-0.001, 0.000642]","(-0.001, 1.0]","(-0.001, 0.0064]",0.000000


In [55]:
df[df["customer_id"] == "id_76084154"]

,customer_id,c_yearly_volumen,c_yearly_charged_amount,c_yearly_margin,c_yearly_trx_count,c_yearly_regions_count,c_yearly_stations_count,c_days_since_last_trx,c_yearly_avg_volume_per_trx,c_yearly_charged_amount_per_liter,...,share_customer_region_8_quantile,share_customer_region_9_quantile,share_customer_region_10_quantile,share_customer_region_11_quantile,share_customer_region_12_quantile,share_customer_region_13_quantile,share_customer_region_14_quantile,share_customer_region_15_quantile,share_customer_region_16_quantile,margin_binned
4893,id_76084154,406400.66,431096645.0,3.368886e+07,996,10,62,0,408.032791,1060.767581,...,"(-0.001, 0.00136]","(0.0306, 0.114]","(0.0665, 1.0]","(-0.001, 1.0]","(-0.001, 1.0]","(0.0538, 0.173]","(-0.001, 0.000642]","(-0.001, 1.0]","(-0.001, 0.0064]",0.0
